# TP2 - Informe tecnico

Sistema de Deteccion y Clasificacion de Razas de Perros — IA 5.2 Computer Vision.

## Equipo
- Alumno 1 : Matías Taborda
- Alumno 2 :

## 1. Explicacion completa del pipeline

Pipeline progresivo de computer vision para detectar y clasificar razas de perros. El flujo sigue este orden:

Embeddings → búsqueda por similitud → clasificación supervisada → detección → pipeline completo

**Etapa 1: Búsqueda por similitud**
Usé una ResNet50 preentrenada en ImageNet (sin fine-tuning) como extractor de características. Para cada imagen del dataset saqué un embedding de 2048 dimensiones. Estos embeddings los guardé en PostgreSQL con la extensión pgvector. Cuando entra una imagen de consulta, calcula su embedding y recupera los K vecinos más cercanos mediante similitud coseno y definí la raza por voto mayoritario.

**Etapa 2: Clasificación supervisada**
Utilicé un modelo de clasificación haciendo fine-tuning de una ResNet18 preentrenada en ImageNet. Congelé las capas convolucionales y cambié únicamente la capa FC final para adaptarla a las 70 razas del dataset. También armé una CNN propia (CNNCustom) para utilizarla como Modelo B. Luego exporté los modelos entrenados como un checkpoint .pth, quedando expuestos a través de ClassifierService.

**Etapa 3: Detección y clasificación**
Integré YOLOv8n para la detección de objetos. El modelo filtra y se queda solo con las detecciones de la clase dog. Por cada bounding box detectado, recorté la región de la imagen y se la pasé al clasificador diseñado en la etapa 2 para identificar la raza. El output final devuelve la imagen anotada con sus boxes, las etiquetas de la raza y los scores de confianza.

**Integración**
Los tres servicios implementados en los .py (SimilarityService, ClassifierService, DetectionService) corren bajo una API en FastAPI montada en Docker. El frontend en Gradio hace requests HTTP a la API en localhost:8080. Luego, la base vectorial (PostgreSQL/pgvector) corre en un contenedor separado con volumen persistente para no perder la data.

## 2. Dataset

Usé el dataset **70 Dog Breeds Image Dataset** de Kaggle, organizado en carpetas por clase.

### Distribución general

| Split | Imágenes totales |
|-------|-----------------|
| Train | 7946 |
| Valid | 700 |
| Test | 700 |
| Total | 9346 |

71 clases en total. En train la distribución es moderadamente desbalanceada: la raza más representada es Shih-Tzu con 198 imágenes y la menos representada es American Hairless con 65 imágenes. Luego, valid y test tienen exactamente 10 imágenes por raza.

### Splits

La estructura de carpetas define los splits:
- `data/dataset/train/`: entrenamiento y ajuste de pesos
- `data/dataset/valid/`: monitoreo durante el entrenamiento
- `data/dataset/test/`: evaluación final

### Conjunto independiente de evaluación

Probé el sistema con imágenes descargadas de internet (por ejemplo, Chihuahua o Golden Retriever) tomadas en condiciones distintas a las del dataset (fondo distinto, iluminación distinta, etc).

### Calidad del dataset

No fue necesario aplicar filtros adicionales al dataset de Kaggle (bien curado), verifiqué que las imágenes tengan buena resolución y los perros sean identificables.

## 3. Preprocesamiento

Se aplicaron las siguientes transformaciones, diferenciando entre train y validación/test:

Pipeline de entrenamiento

In [ ]:
T.Compose([
    T.Resize(256),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Pipeline de validación / test / inferencia

In [ ]:
T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

### Train
- **Resize(256)**: escala manteniendo proporción de aspecto.
- **RandomHorizontalFlip**: un perro mirando a la derecha o izquierda es la misma raza. Duplica efectivamente el dataset.
- **RandomRotation(15°)**: simula variaciones de ángulo en fotos reales.
- **ColorJitter(brightness=0.2, contrast=0.2)**: simula condiciones de iluminación distintas entre tomas.
- **CenterCrop(224)**: recorte central estándar de ImageNet.
- **ToTensor + Normalize(media y std de ImageNet)**: requerido por los modelos preentrenados.

### Validación y test
- **Resize(256) + CenterCrop(224)**: mismo encuadre, sin augmentation.
- **ToTensor + Normalize**: idéntico al pipeline de train.

### Justificación
- El data augmentation se aplica únicamente en train para aumentar la variabilidad artificial sin contaminar la evaluación y las transformaciones son las estándar para fine-tuning sobre ImageNet.

- No apliqué blur ni ruido aleatorio. Para clasificar razas, las características clave son la textura del pelaje y la estructura facial, y ambas se degradan con blur y ruido. Dado que el modelo alcanzó 93.3% de accuracy con buena convergencia, no fue necesario incorporarlos como regularización adicional.

## 4. Justificacion de los modelos elegidos

**Etapa 1: ResNet50 (extractor de embeddings)**
Elegí ResNet50 preentrenado en ImageNet como extractor de características. Al reemplazar la FC por `nn.Identity()`, la salida es un vector de 2048 dimensiones que representa el contenido visual de la imagen. No requiere entrenamiento adicional y genera embeddings con alta separación semántica entre razas, lo que explica el NDCG@10 = 0.9671 que obtuve.

**Trade-offs:**
- Precisión: alta (NDCG@10: 0.9671)
- Velocidad: moderada (aproxc. 200ms por imagen en CPU)
- Memoria: alrededor de 100MB
- Complejidad: nula, sin entrenamiento

**Etapa 2: ResNet18 fine-tuned**
Elegí ResNet18 por ser más liviana que ResNet50 manteniendo buena precisión. Congelé todas las capas convolucionales y entrené solo la FC final adaptada a 70 clases (~36K parámetros).

**Trade-offs:**
- Precisión: 93.3% accuracy en test
- Velocidad: rápida (aprox. 50ms por imagen)
- Memoria: aprox. 45MB
- Complejidad: baja

**Etapa 3: YOLOv8n (para detección)**
Elegí YOLOv8n preentrenado en COCO para la detección de perros. YOLO filtra la clase `dog` (id 16) y devuelve bounding boxes con score de confianza, además no requiere entrenamiento adicional.

**Trade-offs:**
- Precisión: alta detección en escenas complejas
- Velocidad: muy rápida, diseñada para tiempo real
- Memoria: aprox. 6MB
- Complejidad: nula

## 5. Proceso de entrenamiento e hiperparametros

### Modelo A: ResNet18 fine-tuned

**Estrategia:** elegí como estrategia feature extraction, congelando todas las capas convolucionales y solo entrené solo la capa FC final.

**Hiperparámetros:**
| Parámetro | Valor |
|-----------|-------|
| Epochs | 10 |
| Batch size | 32 |
| Learning rate | 1e-3 |
| Optimizador | Adam |
| Criterio | CrossEntropyLoss |
| Image size | 224×224 |

**Proceso:**
- Cargué ResNet18 con pesos de ImageNet
- Congelé todos los parámetros del backbone
- Reemplacé la FC final por `nn.Linear(512, 70)`
- Entrené solo la nueva capa durante 10 épocas
- Guardé el checkpoint en `models/resnet18_finetuned.pth`

**Curvas de entrenamiento:**
La accuracy de validación alcanzó aprox. 90% y convergió con la de train, indicando buena generalización, mientras que la loss de validación se estabilizó a partir de la época 4 mientras la de train seguía bajando, lo que sugiere leve overfitting. Para mitigarlo se podría aplicar early stopping o descongelar algunas capas del backbone en una segunda fase.

### Checkpoints disponibles
- `resnet18_finetuned.pth`: https://drive.google.com/file/d/1ckj8IBtjuh1tJjLGeA5IBb62nxgErNE1/view?usp=sharing
- `cnn_custom.pth`: https://drive.google.com/file/d/1yU-kkF7mtEVwzpZWSNiZJZ2coYzUKJ24/view

## 6. Resultados obtenidos

- Etapa 1: NDCG@10 y justificacion del resultado.
- Etapa 2: accuracy, precision, recall, specificity, F1, matriz de confusion.

In [ ]:
import os
os.environ["EMBEDDING_DIM"] = "2048"
os.environ["USE_PGVECTOR"] = "true"
os.environ["POSTGRES_HOST"] = "localhost"
os.environ["POSTGRES_PORT"] = "5432"
os.environ["POSTGRES_DB"] = "dogs"
os.environ["POSTGRES_USER"] = "dogs_user"
os.environ["POSTGRES_PASSWORD"] = "dogs_pass"

from importlib import reload
import lib.config
reload(lib.config)
from lib.config import settings

from lib.bootstrap import build_similarity, build_store
store = build_store(settings)
similarity = build_similarity(settings, store)

print(settings.embedding_dim)  # 2048 en src/.env y .env.docker.example

scores = []
for breed_dir in sorted(dataset_path.iterdir()):
    if not breed_dir.is_dir():
        continue
    for img_path in list(breed_dir.iterdir())[:10]:
        image = similarity._load_image(str(img_path))
        embedding = similarity.extract_embedding(image)
        neighbors = similarity.search_similar_images(embedding, top_k=10)
        relevance = [1 if n.breed == breed_dir.name else 0 for n in neighbors]
        scores.append(ndcg_at_k(relevance, k=10))

print(f"NDCG@10: {np.mean(scores):.4f}")


INFO:lib.bootstrap:Using PostgreSQL vector store


2048
NDCG@10: 0.9671


**Etapa 1: Búsqueda por similitud**

**NDCG@10: 0.9671**

Un valor cercano a 1.0 indica que el espacio de embeddings de 2048 dimensiones tiene buena separación semántica entre razas.

**Etapa 2: Clasificación supervisada**

| Métrica | Valor |
|---------|-------|
| Accuracy | 0.9329 |
| Precision | 0.9449 |
| Recall | 0.9329 |
| Specificity | 0.9990 |
| F1 | 0.9321 |

Alcancé 93.3% de accuracy en test entrenando solo la FC final durante 10 épocas. Luego, la especificidad de 99.9% indica que el modelo raramente confunde razas, y los errores más frecuentes aparecen entre razas con características físicas similares como spaniels y terriers.

## 7. Comparacion entre enfoques

### Búsqueda por similitud (Etapa 1) vs Clasificación supervisada (Etapa 2)

| | Búsqueda por similitud | Clasificación supervisada |
|--|----------------------|--------------------------|
| Métrica | NDCG@10: 0.9671 | Accuracy: 93.3% |
| Entrenamiento | No requiere | Fine-tuning necesario |
| Resultado | Raza + imágenes similares | Solo raza |
| Escalabilidad | Depende del tamaño del índice | Independiente del dataset |
| Interpretabilidad | Alta (muestra ejemplos visuales) | Baja (caja negra) |

- La búsqueda por similitud no requiere entrenamiento devuelve ejemplos visuales que podemos comparar, pero su rendimiento depende del tamaño y calidad del índice. 
- La clasificación supervisada es más directa y escalable, pero requiere entrenamiento y no explica el resultado.

### ResNet18 fine-tuned vs CNN custom

| Métrica | ResNet18 FT | CNN Custom |
|---------|-------------|------------|
| Accuracy | 0.9386 | 0.4757 |
| Precision | 0.9480 | 0.5118 |
| Recall | 0.9386 | 0.4757 |
| Specificity | 0.9991 | 0.9924 |
| F1 | 0.9378 | 0.4570 |
| Épocas | 10 | 20 |
| Transfer learning | Sí | No |

### Clases con peor desempeño de CNN Custom
| Raza | Precision | Recall | F1 |
|------|-----------|--------|----|
| Boston Terrier | 0.750 | 0.30 | 0.429 |
| Bulldog | 0.474 | 0.90 | 0.621 |
| French Bulldog | 0.857 | 0.60 | 0.706 |
| Shih-Tzu | 1.000 | 0.60 | 0.750 |
| Cockapoo | 0.727 | 0.80 | 0.762 |
| Shiba Inu | 0.875 | 0.70 | 0.778 |
| Dingo | 0.800 | 0.80 | 0.800 |
| Greyhound | 0.714 | 1.00 | 0.833 |
| Lhasa | 0.714 | 1.00 | 0.833 |
| Labradoodle | 0.889 | 0.80 | 0.842 |

### Análisis de errores
En la CNN custom, las clases con peor rendimiento muestran dos problemas claros:

* Alto recall, baja precision: lo vemos en razas como Bulldog, Greyhound o Lhasa. El modelo las tira por descarte cuando duda, lo que genera muchos falsos positivos.

* Bajo recall, alta precision (cuesta detectarlas): se observa en razas como Boston Terrier o Shih-Tzu. Cuando el modelo las predice, acierta. El problema es que sin preentrenamiento le cuesta demasiado identificarlas de entrada.

Un ejemplo sería la confusión entre el Boston Terrier, el French Bulldog y el Bulldog. Como comparten características físicas, la CNN custom los confunde, lo que sugiere que le falta el nivel de detalle para hilar fino que te da ImageNet.

### Trade-offs
ResNet18 (Fine-Tuned) le saca aprox. 46 puntos de accuracy a la CNN custom. Esto indica que el transfer learning es clave si tenemos pocos datos (algo como 8000 imágenes).

Además, ResNet18 converge el doble de rápido (en 10 épocas vs. 20) y solo entreno 35k parámetros contra 1.1M de la custom, lo que la hace más rápida y menos propensa a overfittear. A mi parecer, la ventaja principal de la CNNE custom sería que al ser propia, no depende de checkpoints externos y permite modificar su arquitectura más facilmente.

## 8. Problemas encontrados y soluciones implementadas

### EMBEDDING_DIM inconsistente entre entornos
El `.env.docker.example` tenía `EMBEDDING_DIM=512` mientras que `src/.env` tenía `EMBEDDING_DIM=2048` (ya se lo había cambiado previamente), por lo que el backend de Docker borraba y recreaba la tabla de embeddings cada vez que arrancaba, dejando el índice vacío. Lo resolví simplemente haciendo que los dos archivos tuvieran el mismo `EMBEDDING_DIM=2048`.

### Modelo cargado en cada llamada a extract_embedding
Inicialmente, la implementación cargaba ResNet50 en cada llamada a `extract_embedding`, lo que generaba una latencia o "lag" de 2-3 segundos por imagen. Lo que hice fue inicializar el modelo una sola vez en `__init__` de `SimilarityService`.

## 9. Modificaciones fuera de las funciones indicadas

**Inicialización de ResNet50 en `__init__` de `SimilarityService`**

Cargué el modelo ResNet50 dentro del método __init__ de "SimilarityService.ipynb".
La idea es inicializar el modelo una sola vez en lugar de cargarlo cada llamada a extract_embedding. 

**`train_classifier` retorna el historial de entrenamiento**

Modifiqué la firma de `train_classifier` de `-> None` a `-> dict` y agregué el seguimiento de métricas por época (`train_acc`, `val_acc`, `train_loss`, `val_loss`), con el fin de graficar las curvas de entrenamiento en la notebook de Colab, tal como lo pide el enunciado de la etapa 2.